# Pytorch Tests (multiple modules verify your own support by running)

## TEST LLM

## Load Dataset

We need a large text dataset. Let's use the Hugging Face "wikitext" dataset (Wikipedia articles).

In [ ]:
from datasets import load_dataset

# Load Wikitext dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Get only the "train" portion
train_text = " ".join(dataset["train"]["text"])
print(train_text[:500])  # Print first 500 characters


## Tokenization

LLMs don’t process raw text. Instead, they tokenize text into numbers.
We'll use the GPT-2 tokenizer for this.
The prompt is split into smaller pieces called tokens (e.g., words or subwords).
Example: "How do black holes work?" → [How, do, black, holes, work, ?]


In [ ]:
from transformers import GPT2Tokenizer

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Set the padding token
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 does not have a pad token, so we use eos_token

# Tokenize dataset with padding
tokens = tokenizer(["Hello, how are you?", "I'm fine, thank you!"], 
                   return_tensors="pt", truncation=True, padding=True)

print(tokens["input_ids"].shape)  # Check token shape


## Build Transformer Model (Mini-GPT)

let's define a simple GPT-style model using PyTorch.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import GPT2Model

class MiniGPT(nn.Module): # extends nn.Module
    def __init__(self):
        super(MiniGPT, self).__init__() # Call parent class constructor (nn.Module)
        self.gpt = GPT2Model.from_pretrained("gpt2")  # Load GPT-2 model
        self.fc = nn.Linear(768, tokenizer.vocab_size)  # Output layer

    def forward(self, input_ids):
        output = self.gpt(input_ids).last_hidden_state  # Transformer processing
        return self.fc(output)  # Convert to token probabilities

# Create model
model = MiniGPT()
print(model)


## Train the Model

We train the model using a loss function (CrossEntropy) and backpropagation.

In [ ]:
# Training setup
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-5)
loss_fn = nn.CrossEntropyLoss()

# Convert tokens to tensors
input_ids = tokens["input_ids"].to(device)
labels = input_ids.clone()  # Predict the same text

# Training loop (basic example)
for epoch in range(2):  # Train for 2 epochs
    optimizer.zero_grad()
    outputs = model(input_ids)  # Forward pass
    loss = loss_fn(outputs.view(-1, tokenizer.vocab_size), labels.view(-1))  # Compute loss
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}, Loss: {loss.item()}")


## Generate Text (Inference)

After training, let's generate text based on a user prompt.

In [ ]:
from transformers import GPT2LMHeadModel

# Load GPT-2 model for text generation
gen_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

def generate_text(prompt, max_length=100):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    output = gen_model.generate(input_ids, max_length=max_length)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Example usage
print(generate_text("Once upon a time"))
